In [1]:
// load the formatting setup, the visualization functions, and the style sheet, install Z3 if necessary
#load "../setup.fsx"
#load "../visuals.fsx"
installZ3 ()
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

In [2]:
// load Z3
#load "../z3.fsx"
open Microsoft.Z3

# Sudoku

The sudoku we want to solve is shown here:
    <img src="sudoku.png" width="40%">
I have taken it from https://sudoku.zeit.de/sudoku-hoellisch.

The puzzle is represented as an array of rows, where every row is an array of digits.  Empty fields are represented by the digit `0`.

In [3]:
let puzzle : int[][] =
    [| [| 0; 3; 9; 0; 0; 0; 0; 0; 7 |]
       [| 0; 0; 0; 7; 0; 0; 4; 9; 2 |]
       [| 0; 0; 0; 0; 6; 5; 0; 8; 3 |]
       [| 0; 0; 0; 6; 0; 3; 2; 7; 0 |]
       [| 0; 0; 0; 0; 4; 0; 8; 0; 0 |]
       [| 5; 6; 0; 0; 0; 0; 0; 0; 0 |]
       [| 0; 0; 5; 2; 0; 9; 0; 0; 1 |]
       [| 0; 2; 1; 0; 0; 0; 0; 4; 0 |]
       [| 7; 0; 0; 0; 0; 0; 5; 0; 0 |] |]

We are going to solve this puzzle with the help of the constraint solver `Z3`.  For every field of the Sudoku we use an integer variable.  `Variables[row][col]` is the variable for the field in the given row and column, where rows and columns are numbered starting from `0`.  The name of this variable is `Vrc`, where `r` and `c` are numbered starting from `1`.

In [4]:
let Variables = [| for row in 0 .. 8 -> [| for col in 0 .. 8 -> Int (sprintf "V%d%d" (row + 1) (col + 1)) |] |]

The function `constraintsFromPuzzle` takes one argument:
* `Variables` is the matrix of `Z3` variables described above.

It returns a list of constraints specifying that the variables corresponding to numbers that are already given in the Sudoku take the specified values.

In [5]:
let constraintsFromPuzzle (Variables: IntExpr[][]) : BoolExpr list =
    [ for row in 0 .. 8 do
        for col in 0 .. 8 do
            if puzzle[row][col] <> 0 then
                Variables[row][col] .= puzzle[row][col] ]

In [6]:
constraintsFromPuzzle Variables

[V12 = 3; V13 = 9; V19 = 7; V24 = 7; V27 = 4; V28 = 9; V29 = 2; V35 = 6; V36 = 5; V38 = 8; V39 = 3; V44 = 6; V46 = 3; V47 = 2; V48 = 7; V55 = 4; V57 = 8; V61 = 5; V62 = 6; V73 = 5; V74 = 2; V76 = 9; V79 = 1; V82 = 2; V83 = 1; V88 = 4; V91 = 7; V97 = 5]

The function `allConstraints` takes the matrix `Variables` as its argument.  It returns a list of constraints that encodes the given Sudoku:
- the digits given in the puzzle are respected,
- all entries in a row are different,
- all entries in a column are different,
- all entries in a $3 \times 3$ square are different,
- all entries are digits between 1 and 9.

The function `Distinct` is defined in `z3.fsx`.  It takes a list of expressions and returns a formula that is true iff all these expressions have different values.

In [7]:
let allConstraints (Variables: IntExpr[][]) : BoolExpr list =
    let L = [ 0 .. 8 ]
    let mutable Constraints = constraintsFromPuzzle Variables
    // all entries in a row are unique
    Constraints <- Constraints @ [ for row in L -> Distinct [ for col in L -> Variables[row][col] ] ]
    // all entries in a column are unique
    Constraints <- Constraints @ [ for col in L -> Distinct [ for row in L -> Variables[row][col] ] ]
    // all entries in a 3x3 square are unique
    Constraints <- Constraints @ [ for r in 0 .. 2 do
                                     for c in 0 .. 2 ->
                                       Distinct [ for row in 0 .. 2 do
                                                    for col in 0 .. 2 -> Variables[r*3 + row][c*3 + col] ] ]
    // all numbers are between 1 and 9
    Constraints <- Constraints @ [ for row in L do
                                     for col in L do
                                       yield 0 .< Variables[row][col]
                                       yield Variables[row][col] .< 10 ]
    Constraints

In [8]:
(allConstraints Variables).Length

217

The function `solve` receives no arguments.  It computes a solution of the Sudoku and returns this solution as a map that assigns the digits to the positions `(row, col)`, where rows and columns are numbered starting from `1`.  If the Sudoku can not be solved, a message is printed and the empty map is returned.  Note that this function hides the function `solve` from `z3.fsx`.

In [9]:
let solve () : Map<int * int, int> =
    let S = newSolver ()
    S.Add(Array.ofList (allConstraints Variables))
    match S.Check() with
    | Status.SATISFIABLE ->
        let M = S.Model
        Map [ for row in 0 .. 8 do
                for col in 0 .. 8 -> ((row + 1, col + 1), intValue M (Variables[row][col])) ]
    | Status.UNSATISFIABLE ->
        printfn "The problem is not solvable."
        Map.empty
    | _ ->
        printfn "Z3 cannot determine whether the problem is solvable."
        Map.empty

In [10]:
#!time
let Solution = solve ()
Solution

map [((1, 1), 1); ((1, 2), 3); ((1, 3), 9); ((1, 4), 4); ((1, 5), 2); ((1, 6), 8); ((1, 7), 6); ((1, 8), 5); ((1, 9), 7); ((2, 1), 6); ((2, 2), 5); ((2, 3), 8); ((2, 4), 7); ((2, 5), 3); ((2, 6), 1); ((2, 7), 4); ((2, 8), 9); ((2, 9), 2); ((3, 1), 2); ((3, 2), 4); ((3, 3), 7); ((3, 4), 9); ((3, 5), 6); ((3, 6), 5); ((3, 7), 1); ((3, 8), 8); ((3, 9), 3); ((4, 1), 8); ((4, 2), 1); ((4, 3), 4); ((4, 4), 6); ((4, 5), 9); ((4, 6), 3); ((4, 7), 2); ((4, 8), 7); ((4, 9), 5); ((5, 1), 9); ((5, 2), 7); ((5, 3), 3); ((5, 4), 5); ((5, 5), 4); ((5, 6), 2); ((5, 7), 8); ((5, 8), 1); ((5, 9), 6); ((6, 1), 5); ((6, 2), 6); ((6, 3), 2); ((6, 4), 1); ((6, 5), 8); ((6, 6), 7); ((6, 7), 9); ((6, 8), 3); ((6, 9), 4); ((7, 1), 4); ((7, 2), 8); ((7, 3), 5); ((7, 4), 2); ((7, 5), 7); ((7, 6), 9); ((7, 7), 3); ((7, 8), 6); ((7, 9), 1); ((8, 1), 3); ((8, 2), 2); ((8, 3), 1); ((8, 4), 8); ((8, 5), 5); ((8, 6), 6); ((8, 7), 7); ((8, 8), 4); ((8, 9), 9); ((9, 1), 7); ((9, 2), 9); ((9, 3), 6); ((9, 4), 3); ((9, 5)

Wall time: 57.3021ms

## Graphical Representation

The function `showSolution` takes two arguments:
- `Solution` is a map assigning the digits to the positions `(row, col)` of the Sudoku,
- `width` is the width of the grid as a CSS length.

It displays the Sudoku.  The digits that were given in the puzzle are shown in black, the digits computed by the solver are shown in blue.  The function `sudokuGrid` is defined in the file `visuals.fsx`.

In [11]:
let showSolution (Solution: Map<int * int, int>) (width: string) =
    sudokuGrid puzzle Solution width

In [12]:
showSolution Solution "50%"

1,3,9,4,2,8,6,5,7
6,5,8,7,3,1,4,9,2
2,4,7,9,6,5,1,8,3
8,1,4,6,9,3,2,7,5
9,7,3,5,4,2,8,1,6
5,6,2,1,8,7,9,3,4
4,8,5,2,7,9,3,6,1
3,2,1,8,5,6,7,4,9
7,9,6,3,1,4,5,2,8
